In [4]:
import os
import random
import unicodedata
import pandas as pd

In [5]:
templates_data = pd.read_csv('personal/sentence_template_starter.csv')
towns_data = pd.read_csv('personal/french_town_start.csv')
generated_data = pd.read_csv('personal/generated_french_town_dataset.csv')


In [6]:
templates_valid = templates_data[templates_data['label'] == 'VALID']
templates_invalid = templates_data[templates_data['label'] == 'INVALID']
both_templates = pd.concat([templates_valid, templates_invalid])
towns = towns_data['nom_ville'].tolist()

In [7]:
generated_data = []
total_data_number = 0

valid_number = random.randrange(1500, 2000)
invalid_number = random.randrange(1500, 2000)
lower_valid = int(valid_number * 0.15)
lower_invalid = int(invalid_number * 0.15)
without_accent = random.randrange(400, 600)

if generated_data:
    os.remove('personal/generated_french_town_dataset.csv')


In [8]:
def generate_sentence(template, town1, town2, should_lowercase):
    if should_lowercase:
        template = template.lower()
        town1_str = str(town1).lower()
        town2_str = str(town2).lower()
    else:
        town1_str = str(town1)
        town2_str = str(town2)

    return template.replace('{ville1}', town1_str).replace('{ville2}', town2_str)


def generate_data_for_label(templates_df, count, lower_count, label, start_id):
    data = []
    for i in range(count):
        row = templates_df.sample(1).iloc[0]
        town1, town2 = random.sample(towns, 2)
        should_lowercase = i < lower_count
        sentence = generate_sentence(row['template'], town1, town2, should_lowercase)

        if random.random() < 0.1:
            sentence = add_noise_to_sentence(sentence)

        data.append({
            'sentence_id': start_id + i,
            'label': label,
            'sentence': sentence,
            'town1': town1,
            'town2': town2,
            'template_id': row['template_id']
        })

    return data


def generate_data_without_accent(templates_df, count, start_id):
    data = []
    lower_count = count // 2

    for i in range(count):
        row = templates_df.sample(1).iloc[0]
        town1, town2 = random.sample(towns, 2)
        should_lowercase = i < lower_count

        sentence = generate_sentence(row['template'], town1, town2, should_lowercase)
        sentence_no_accent = unicodedata.normalize('NFD', sentence).encode('ascii', 'ignore').decode('utf-8')

        if random.random() < 0.1:
            sentence_no_accent = add_noise_to_sentence(sentence_no_accent)

        data.append({
            'sentence_id': start_id + i,
            'label': row['label'],
            'sentence': sentence_no_accent,
            'town1': town1,
            'town2': town2,
            'template_id': row['template_id']
        })

    return data


def add_noise_to_sentence(sentence, noise_level=0.05):
    sentence = list(sentence)

    for i in range(len(sentence)):
        if random.random() < noise_level:
            operator = random.choice(['delete', 'swap', 'duplicate'])

            if operator == 'delete' and len(sentence) > 1:
                sentence[i] = ''
            elif operator == 'swap' and i < len(sentence) - 1:
                sentence[i], sentence[i + 1] = sentence[i + 1], sentence[i]

            elif operator == 'duplicate':
                sentence[i] = sentence[i] * 2

    noisy_sentence = ''.join(sentence)

    if random.random() < noise_level:
        noisy_sentence = ' ' + noisy_sentence
    if random.random() < noise_level:
        noisy_sentence = noisy_sentence + ' '

    return noisy_sentence

In [9]:
generated_data = []

valid_data = generate_data_for_label(
    templates_valid,
    valid_number,
    lower_valid,
    'VALID',
    start_id=1
)

invalid_data = generate_data_for_label(
    templates_invalid,
    invalid_number,
    lower_invalid,
    'INVALID',
    start_id=len(valid_data) + 1
)

accent_data = generate_data_without_accent(
    both_templates,
    without_accent,
    start_id=len(valid_data) + len(invalid_data) + 1
)

generated_data.extend(valid_data)
generated_data.extend(invalid_data)
generated_data.extend(accent_data)

In [10]:
df_generated = pd.DataFrame(generated_data).sample(frac=1)
df_generated.to_csv('personal/generated_french_town_dataset.csv', index=False)
df_generated.head()

,sentence_id,label,sentence,town1,town2,template_id
3459,3460,INVALID,mon ami vit a pau et moi a angers,Pau,Angers,19
544,545,VALID,Je dois me rendre de Concarneau à Perpignan,Concarneau,Perpignan,7
327,328,VALID,Prochain train pour Troyes depuis Montauban,Montauban,Troyes,13
528,529,VALID,Je voudrais partir ed lAbi pour aller à Tourcoing,Albi,Tourcoing,9
1166,1167,VALID,Prochain train pour Maubeuge depuis Lille,Lille,Maubeuge,13
